# Spark with MongoDB and MySQL

This notebook demonstrates how to read from and write to MongoDB and MySQL using PySpark.

In [1]:
from pyspark.sql import SparkSession

# Initialize SparkSession with required packages
spark = SparkSession.builder \
    .appName("DB Test") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0,mysql:mysql-connector-java:8.0.33") \
    .getOrCreate()

sc = spark.sparkContext

## MongoDB Test

In [2]:
# Create a DataFrame
data = [("Alice", 34), ("Bob", 45), ("Charlie", 29)]
columns = ["Name", "Age"]
df = spark.createDataFrame(data, columns)

# Write to MongoDB
df.write \
    .format("mongodb") \
    .mode("overwrite") \
    .option("connection.uri", "mongodb://mongo:27017/testdb.people") \
    .option("database", "testdb") \
    .option("collection", "people") \
    .save()

print("Written to MongoDB")

Written to MongoDB


In [3]:
# Read from MongoDB
df_mongo = spark.read \
    .format("mongodb") \
    .option("connection.uri", "mongodb://mongo:27017/testdb.people") \
    .option("database", "testdb") \
    .option("collection", "people") \
    .load()

df_mongo.show()

+---+-------+--------------------+
|Age|   Name|                 _id|
+---+-------+--------------------+
| 45|    Bob|691f09f93ecc0f6e4...|
| 34|  Alice|691f09f9a92f50080...|
| 29|Charlie|691f09f99f092b3b2...|
+---+-------+--------------------+



## MySQL Test

In [4]:
# Write to MySQL
jdbc_url = "jdbc:mysql://mysql:3306/testdb"
properties = {
    "user": "root",
    "password": "root",
    "driver": "com.mysql.cj.jdbc.Driver"
}

df.write \
    .jdbc(url=jdbc_url, table="people", mode="overwrite", properties=properties)

print("Written to MySQL")

Written to MySQL


In [5]:
# Read from MySQL
df_mysql = spark.read \
    .jdbc(url=jdbc_url, table="people", properties=properties)

df_mysql.show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 34|
|    Bob| 45|
|Charlie| 29|
+-------+---+



In [6]:
spark.stop()